In [ ]:
%pip install python-docx
%pip install paddleocr
%pip install pillow

In [7]:
%pip install pytesseract

  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install pillow python-docx

Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install opencv-python pillow pytesseract numpy

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
    --------------------------------------- 0.5/40.2 MB 430.2 kB/s eta 0:01:33
    --------------------------------------- 0.5/40.2 MB 430.2 kB/s eta 0:01:33
    --------------------------------------- 0.8/40.2 MB 492.0 kB/s eta 0:01:21
    --------------------------------------- 0.8/40.2 MB 492.0 kB/s eta 0:01:21
    --------------------------------------- 0.8/40.2 MB 492.0 kB/s eta 0:01:21
    --------------------------------------- 0.8/40.2 MB 492.0 kB/s eta 0:01:21
    --------------------------------------- 0.8/40.2 MB 492.0 kB/s eta 0:01:21
    -----------------------

ERROR: Could not install packages due to an OSError: [WinError 5] 액세스가 거부되었습니다: 'c:\\Users\\호두마루\\Desktop\\project\\communication-law\\venv\\Lib\\site-packages\\cv2\\cv2.pyd'
Check the permissions.



In [14]:
import os
import io
import zipfile
from typing import List

import docx
import pytesseract
from PIL import Image


# -----------------------------
# 설정
# -----------------------------
INPUT_DOCX = r"./접구선통기술기준_20251128.docx"
OUTPUT_DOCX = r"./접구선통기술기준_20251128_markdown.docx"

# Windows에서 Tesseract 설치 경로 지정
# 본인 PC 경로에 맞게 수정하세요.
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


# -----------------------------
# DOCX 내부 이미지 추출
# -----------------------------
def extract_images_from_docx(docx_path: str) -> List[Image.Image]:
    images = []

    with zipfile.ZipFile(docx_path, "r") as z:
        for name in z.namelist():
            if name.startswith("word/media/"):
                image_bytes = z.read(name)
                img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
                images.append(img)

    return images


# -----------------------------
# OCR 결과를 Markdown으로 변환
# -----------------------------
def image_to_markdown(img: Image.Image) -> str:
    text = pytesseract.image_to_string(img, lang="kor+eng")

    # OCR 결과를 줄 단위로 정리
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if not lines:
        return "[이미지 OCR 결과 없음]"

    # 간단한 Markdown 표 형태
    md_lines = ["| 내용 |", "|---|"]
    for line in lines:
        line = line.replace("|", " ")
        md_lines.append(f"| {line} |")

    return "\n".join(md_lines)


# -----------------------------
# DOCX의 이미지 문단을 Markdown으로 치환
# -----------------------------
def replace_image_paragraphs_with_markdown(input_docx: str, output_docx: str) -> None:
    if not os.path.exists(input_docx):
        raise FileNotFoundError(f"입력 파일이 없습니다: {input_docx}")

    doc = docx.Document(input_docx)
    images = extract_images_from_docx(input_docx)

    if not images:
        print("DOCX 내부에서 이미지를 찾지 못했습니다.")
        doc.save(output_docx)
        return

    image_index = 0

    for para in doc.paragraphs:
        xml = para._element.xml

        # 문단 안에 그림이 있는지 대략 판별
        if "graphic" in xml or "pic:pic" in xml:
            if image_index >= len(images):
                break

            md = image_to_markdown(images[image_index])

            # 기존 문단 비우기
            for run in para.runs:
                run.text = ""

            para.clear()
            para.add_run(md)

            image_index += 1

    doc.save(output_docx)
    print(f"완료: {output_docx}")


# -----------------------------
# 실행
# -----------------------------
if __name__ == "__main__":
    replace_image_paragraphs_with_markdown(INPUT_DOCX, OUTPUT_DOCX)

완료: ./접구선통기술기준_20251128_markdown.docx


In [16]:
import io
import os
import re
import zipfile
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
import pytesseract
from PIL import Image


# =========================
# 설정
# =========================
INPUT_DOCX = r"접구선통기술기준_20251128.docx"
OUTPUT_MD = r"output_images.md"
EXTRACT_DIR = r"extracted_images"

# Windows Tesseract 경로
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

OCR_LANG = "kor+eng"


# =========================
# DOCX 내부 이미지 추출
# =========================
def extract_images_from_docx(docx_path: str, output_dir: str) -> List[str]:
    os.makedirs(output_dir, exist_ok=True)
    extracted_files = []

    with zipfile.ZipFile(docx_path, "r") as z:
        media_files = [n for n in z.namelist() if n.startswith("word/media/")]
        media_files.sort()

        for i, name in enumerate(media_files, start=1):
            data = z.read(name)
            ext = Path(name).suffix.lower() or ".png"
            out_path = os.path.join(output_dir, f"image_{i:03d}{ext}")
            with open(out_path, "wb") as f:
                f.write(data)
            extracted_files.append(out_path)

    return extracted_files


# =========================
# OCR용 전처리
# =========================
# def load_image_cv(image_path: str) -> np.ndarray:
#     img = cv2.imread(image_path)
#     if img is None:
#         raise ValueError(f"이미지를 열 수 없습니다: {image_path}")
#     return img
def load_image_cv(image_path: str) -> np.ndarray:
    try:
        pil_img = Image.open(image_path).convert("RGB")
        img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        return img
    except Exception as e:
        raise ValueError(f"이미지를 열 수 없습니다: {image_path} / 원인: {e}")

        
def preprocess_for_table_detection(img: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)

    # 표선 검출을 위한 이진화
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 31, 12
    )
    return gray, binary


# =========================
# 표 구조 검출
# =========================
def detect_table_mask(binary: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    h, w = binary.shape

    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(20, w // 20), 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, h // 20)))

    horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel, iterations=1)

    table_mask = cv2.add(horizontal, vertical)
    return horizontal, vertical, table_mask


def extract_cell_boxes(table_mask: np.ndarray) -> List[Tuple[int, int, int, int]]:
    contours, _ = cv2.findContours(table_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 40 and h > 18:
            boxes.append((x, y, w, h))

    # 중복/바깥 박스 줄이기
    boxes = sorted(boxes, key=lambda b: (b[1], b[0]))
    filtered = []
    for box in boxes:
        x1, y1, w1, h1 = box
        keep = True
        for fx, fy, fw, fh in filtered:
            if x1 >= fx and y1 >= fy and x1 + w1 <= fx + fw and y1 + h1 <= fy + fh:
                keep = False
                break
        if keep:
            filtered.append(box)

    return filtered


def group_boxes_into_rows(boxes: List[Tuple[int, int, int, int]], y_threshold: int = 18):
    if not boxes:
        return []

    rows = []
    current_row = [boxes[0]]

    for box in boxes[1:]:
        _, y, _, _ = box
        _, prev_y, _, _ = current_row[-1]

        if abs(y - prev_y) <= y_threshold:
            current_row.append(box)
        else:
            rows.append(sorted(current_row, key=lambda b: b[0]))
            current_row = [box]

    if current_row:
        rows.append(sorted(current_row, key=lambda b: b[0]))

    return rows


def is_probable_table(rows: List[List[Tuple[int, int, int, int]]]) -> bool:
    if len(rows) < 2:
        return False
    max_cols = max(len(r) for r in rows)
    return max_cols >= 2


# =========================
# 셀 OCR
# =========================
def ocr_cell(gray: np.ndarray, box: Tuple[int, int, int, int]) -> str:
    x, y, w, h = box
    pad = 2
    crop = gray[max(y + pad, 0):y + h - pad, max(x + pad, 0):x + w - pad]

    if crop.size == 0:
        return ""

    # OCR용 추가 이진화
    crop_bin = cv2.threshold(crop, 180, 255, cv2.THRESH_BINARY)[1]
    pil = Image.fromarray(crop_bin)

    text = pytesseract.image_to_string(
        pil,
        lang=OCR_LANG,
        config="--psm 6"
    )
    return normalize_text(text)


def normalize_text(text: str) -> str:
    text = text.replace("\x0c", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", " ", text)
    return text.strip()


# =========================
# 일반 OCR
# =========================
def ocr_free_text(gray: np.ndarray) -> str:
    pil = Image.fromarray(gray)
    text = pytesseract.image_to_string(
        pil,
        lang=OCR_LANG,
        config="--psm 6"
    )
    return normalize_text(text)


# =========================
# Markdown 생성
# =========================
def rows_to_markdown(rows_text: List[List[str]]) -> str:
    if not rows_text:
        return "[표를 인식하지 못했습니다.]"

    max_cols = max(len(r) for r in rows_text)
    norm_rows = []
    for row in rows_text:
        if len(row) < max_cols:
            row = row + [""] * (max_cols - len(row))
        norm_rows.append(row)

    header = norm_rows[0]
    md = []
    md.append("| " + " | ".join(header) + " |")
    md.append("| " + " | ".join(["---"] * max_cols) + " |")

    for row in norm_rows[1:]:
        md.append("| " + " | ".join(row) + " |")

    return "\n".join(md)


def image_to_markdown(image_path: str) -> str:
    img = load_image_cv(image_path)
    gray, binary = preprocess_for_table_detection(img)
    _, _, table_mask = detect_table_mask(binary)

    boxes = extract_cell_boxes(table_mask)
    rows = group_boxes_into_rows(boxes)

    if is_probable_table(rows):
        rows_text = []
        for row in rows:
            row_text = [ocr_cell(gray, box) for box in row]
            row_text = [t if t else "" for t in row_text]
            rows_text.append(row_text)
        return rows_to_markdown(rows_text)

    # 표가 아니면 일반 OCR
    text = ocr_free_text(gray)
    if not text:
        return "[이미지 OCR 결과 없음]"
    return text


# =========================
# DOCX 전체 변환
# =========================
def convert_docx_images_to_markdown(docx_path: str, output_md: str, extract_dir: str):
    image_paths = extract_images_from_docx(docx_path, extract_dir)

    md_parts = [f"# {Path(docx_path).name} 이미지 Markdown 변환\n"]

    if not image_paths:
        md_parts.append("\n[DOCX 내부 이미지 없음]\n")
    else:
        for i, image_path in enumerate(image_paths, start=1):
            md_parts.append(f"\n## 이미지 {i}\n")
            md_parts.append(f"\n원본 파일: `{Path(image_path).name}`\n")
            md_parts.append("\n```markdown")
            md_parts.append(image_to_markdown(image_path))
            md_parts.append("```\n")

    with open(output_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md_parts))

    print(f"완료: {output_md}")


# =========================
# 실행
# =========================
if __name__ == "__main__":
    convert_docx_images_to_markdown(INPUT_DOCX, OUTPUT_MD, EXTRACT_DIR)

완료: output_images.md
